# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR² dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata object provides the dataset's metadata
print(f"Loaded dataset: {dataset.metadata.name}")
print(dataset.metadata.description)

## 2. Data Overview

Review available record sets, their fields, and the corresponding `@id`s. For clarity, this notebook lists record sets and their fields so that we can reference them by their `@id` as required by Croissant.

In [ ]:
from pprint import pprint

# Get all record sets
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s).\n")
for record_set in record_sets:
    print(f"Record Set Name: {record_set.name}")
    print(f"  Record Set @id: {record_set.id}")
    print(f"  Description: {record_set.description if hasattr(record_set, 'description') else ''}")

    # List fields in each record set
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (id: {field.id}, dataType: {field.data_type})")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. Below, we demonstrate for every record set found.

In [ ]:
dfs = {}
for record_set in dataset.record_sets:
    print(f"Loading record set: {record_set.name} (@id: {record_set.id})")
    # Load records as list of dicts
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    if len(df) == 0:
        print("  [!] No records loaded.")
    else:
        print(f"  Loaded shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
    dfs[record_set.id] = df
    print("")

# For quick reference, use the first record set for further analysis
if len(dataset.record_sets) > 0:
    main_record_set = dataset.record_sets[0]
    main_df = dfs[main_record_set.id]
    print(f"Selected record set '@id': {main_record_set.id}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

In this example, we identify a numeric field (column) by its `@id` and name from the record set fields, then demonstrate filtering and normalization.

In [ ]:
import numpy as np

# Find a numeric field (e.g., Float or Integer)
numeric_field = None
for field in main_record_set.fields:
    if field.data_type in ["Float", "Integer", "Number"]:
        numeric_field = field.id
        numeric_field_name = field.name
        print(f"Using numeric field: {numeric_field_name} (@id: {numeric_field})")
        break

if numeric_field is None:
    print("No numeric field detected for EDA.")
else:
    # Filter out non-numeric or missing values
    df_eda = main_df.copy()

    # If values are stored as strings (possible in some Croissant exports), convert to numeric
    df_eda[numeric_field] = pd.to_numeric(df_eda[numeric_field], errors='coerce')
    threshold = df_eda[numeric_field].quantile(0.75)  # e.g., keep upper quartile
    filtered_df = df_eda[df_eda[numeric_field] > threshold]
    print(f"Filtered {len(filtered_df)} records in '{main_record_set.name}' where {numeric_field_name} (@id: {numeric_field}) > {threshold:.2f}\n")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized values for {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a categorical field
    # Attempt to find a non-numeric, possibly text field
    group_field = None
    for field in main_record_set.fields:
        if field.data_type == 'Text' and field.id != numeric_field:
            group_field = field.id
            group_field_name = field.name
            print(f"Can attempt to group by '{group_field_name}' (@id: {group_field})")
            break

    if group_field and group_field in main_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Mean of {numeric_field_name} by {group_field_name}:")
        display(grouped.head())

## 5. Visualization

Visualize distributions or relationships in the dataset, such as histograms of numeric fields or boxplots grouped by categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.xlabel(f"{numeric_field_name} (@id: {numeric_field})")
    plt.title(f"Distribution of {numeric_field_name}")
    plt.show()

    # If grouping is available, show a boxplot
    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.xlabel(f"{group_field_name} (@id: {group_field})")
        plt.ylabel(f"{numeric_field_name}")
        plt.title(f"{numeric_field_name} by {group_field_name}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and examine the FAIR² dataset using `mlcroissant` by referencing all entities through their `@id` fields. We inspected record sets and fields, filtered and normalized a numeric field, and produced basic visualizations for initial exploration.

Further exploration and more advanced analyses (e.g., statistical testing, machine learning) can follow based on specific research questions, always referencing Croissant entities by their `@id` for robustness and reproducibility.